# HAKE-MER — M1 ablation (H1 vs Step 0 baseline)

**Backbone:** DistilBERT-base · **Module:** M1 only (4 phrases × 32 tokens, additive attention)

**Protocol:** same as Step 0 — batch 16, LR 5e-5, 4 epochs, 3 seeds, best val F1-macro.

1. **Runtime → Factory reset** → **GPU**
2. Run all cells → download zip → archive in repo (see last cell)

Compare test F1-macro to Step 0 DistilBERT **0.486 ± 0.005**.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
!./run_m1_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}

In [ ]:
import json
from pathlib import Path

path = Path("reference/artifacts/m1_distilbert_base_uncased_campaign.json")
c = json.loads(path.read_text(encoding="utf-8"))
print(path.name, "protocol:", c.get("protocol", {}))
print("  epochs logged:", len(c["runs"][0]["history"]))
for k, b in c["test_aggregate"].items():
    print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

In [ ]:
import zipfile
from google.colab import files

slug = "distilbert_base_uncased"
campaign = Path(f"reference/artifacts/m1_{slug}_campaign.json")
out_name = "m1_distilbert_step0.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(campaign, campaign.name)
    for m in sorted(Path("runs").glob(f"{slug}_seed*_m1/metrics.json")):
        zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

## Integrity

1. **File → Download → .ipynb** → `reference/training_records/step_m1/colab/`
2. Unzip + campaign JSON → `reference/artifacts/` (same pattern as Step 0)
3. Send zip to your dev machine or commit on GitHub